In [ ]:

import pandas as pd
from datasets import load_dataset
import re

print("Loading dataset...")
dataset = load_dataset("srikanthgali/ai-text-detection-pile-cleaned", split="train")

def is_essay_like(text):
    if len(str(text).split()) < 200:
        return False
    
    special_chars = len(re.findall(r'[\{\}\[\]\(\)\=\;\<\>\_]', str(text)))
    if special_chars / len(str(text)) > 0.05: 
        return False
        
    code_keywords = ["public static", "import pandas", "def function", "return true"]
    if any(keyword in str(text).lower() for keyword in code_keywords):
        return False

    return True

print(f"Original size: {len(dataset)} rows")
filtered_dataset = dataset.filter(lambda x: is_essay_like(x['text']))
print(f"Filtered 'Essay' size: {len(filtered_dataset)} rows")

df_essays = filtered_dataset.to_pandas()

print(df_essays.head())

df_essays.to_csv("filtered_essay_corpus.csv", index=False)
print("Saved as 'filtered_essay_corpus.csv'. Upload this as a Kaggle Dataset.")

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
try:
    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("HF_TOKEN")

    login(secret_value_0)
    print("Logged in to Hugging Face successfully!")
except Exception as e:
    print(f"Error logging in: {e}")
    print("Make sure you added the secret correctly in Add-ons -> Secrets")

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from tqdm import tqdm


MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print("Loading Model in Float16 (Standard Precision)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,  #
    device_map="auto",
    trust_remote_code=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=600, 
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
    repetition_penalty=1.15
)

try:
    df = pd.read_csv("/kaggle/input/hc3-project-cleaned-essays/filtered_essay_corpus.csv")
    source_essays = df[df['generated'] == 0].sample(500, random_state=42)
except:
    print("Dataset not found. Creating dummy topics.")
    source_essays = pd.DataFrame({'text': ["The history of the Roman Empire is complex...", "Climate change is a pressing issue..."]})

adversarial_data = []

print(f"Generating {len(source_essays)} adversarial essays from scratch...")

for index, row in tqdm(source_essays.iterrows(), total=len(source_essays)):
    topic_snippet = " ".join(row['text'].split()[:15])

    print(topic_snippet)
    prompt = [
    {
        "role": "system",
        "content": "You are a college student writing an essay. Write with high burstiness. Mix very short sentences with long, complex ones. Avoid robotic transitions like 'Furthermore'. Use occasional casual phrasing but try to sound academic. Do not use bullet points."
    },
    {
        "role": "user",
        "content": f"Write a 400-word essay starting with: '{topic_snippet}...'"
    }
]
    
    prompt_formatted = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    
    try:
        outputs = pipe(prompt_formatted)
        generated_text = outputs[0]["generated_text"].split("<|assistant|>")[-1].strip()
        
        adversarial_data.append({
            "text": generated_text,
            "generated": 1,            
            "source": "adversarial_scratch",
            "original_topic_source": index
        })
    except Exception as e:
        continue

df_adv = pd.DataFrame(adversarial_data)
df_adv.to_csv("adversarial_scratch_set.csv", index=False)
print("Saved 'adversarial_scratch_set.csv'. Use this for the final robustness evaluation.")

In [ ]:
import pandas as pd

input_file = "/kaggle/input/genai-project-adversarial-essays/adversarial_scratch_set.csv"
output_file = "adversarial_scratch_set_cleaned.csv"

print(f"Loading {input_file}...")
try:
    df = pd.read_csv(input_file)
except FileNotFoundError:
    print("Error: File not found. Please make sure you ran the generation script first.")
    df = pd.DataFrame({
        'text': ["<|im_start|>system\nPrompt...<|im_end|>\n<|im_start|>assistant\nThis is the actual essay text."],
        'generated': [1],
        'source': ['adversarial_scratch']
    })

def clean_generated_text(text):
    target_tag = "<|im_start|>assistant\n"
    
    if target_tag in str(text):
        return text.split(target_tag)[1].strip()
    
    return text

print("Cleaning text column...")
df_clean = df.copy()
df_clean['text'] = df_clean['text'].apply(clean_generated_text)

df_clean.to_csv(output_file, index=False)

print(f"Success! Cleaned data saved to: {output_file}")
print("-" * 30)
print("Example of cleaned text:")
print(df_clean['text'].iloc[0][:300] + "...") 